# 🧬⚛️ Q-OmicSelect Yol Haritası: Sıfırdan QAOA/VQC ile QUBO Optimizasyonu

## Bu notebook ne öğretecek?
Bu notebook, **hiç kuantum bilgisi olmayan biri** için tasarlanmıştır.  
Adım adım şunları öğreneceksiniz:

1. **QUBO nedir?** — Optimizasyon problemlerini kuantum için nasıl yazarız?
2. **Kuantum kapıları nedir?** — Kuantum bilgisayarın ABC'si
3. **QAOA nedir?** — Kuantum yaklaşık optimizasyon algoritması
4. **VQC nedir?** — Değişken kuantum devresi
5. **Gen seçimi problemi** — Omik verilerden gen seçimini QUBO'ya dönüştürme
6. **Mini Q-OmicSelect** — Her şeyi bir araya getirme

---
## ⚡ Önemli Not
Bu notebook **Google Colab**'da çalışır. Gerçek kuantum donanımı gerekmez —  
her şeyi **simülasyon** ile bilgisayarınızda çalıştıracağız.

---

## 📦 BÖLÜM 0: Kütüphaneleri Yükle

İlk çalıştırmada bu hücre **2-3 dakika** sürebilir. Sabırla bekleyin.

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install qiskit==0.46.0 qiskit-aer==0.13.3 qiskit-algorithms==0.3.0 -q
!pip install matplotlib numpy scipy pandas scikit-learn -q

print("✅ Tüm kütüphaneler yüklendi!")

In [ ]:
# Temel importlar
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Qiskit importları
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import Statevector, Pauli, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler, Estimator

print("✅ Tüm importlar başarılı!")
print("🚀 Öğrenmeye başlayabiliriz!")

---
# 📚 BÖLÜM 1: QUBO Nedir?

## 🧠 Önce Sezgi: Optimizasyon Problemi Nedir?

Düşünün: Elimizde 5 gen var. Bunlardan **en iyi 3'ünü seçmemiz** lazım.  
Her genin bir "puanı" var (ne kadar önemli olduğu).  
Bazı gen çiftleri **birlikte seçilince** bonus veriyor (sinerjik etki).  
Bazı gen çiftleri **birlikte seçilince** ceza var (çakışma).

Bu bir optimizasyon problemi: doğru kombinasyonu bulmak!

## 📐 QUBO'nun Matematiksel Tanımı

**QUBO = Quadratic Unconstrained Binary Optimization**

Türkçesi: **İkili Kısıtsız Karesel Optimizasyon**

- **Binary (İkili)**: Her değişken 0 veya 1 (gen seçildi / seçilmedi)
- **Quadratic (Karesel)**: Hem tekli hem de ikili terimler var
- **Unconstrained (Kısıtsız)**: Kısıtları da formüle dahil ediyoruz

$$\text{minimize} \quad E(x) = \sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i x_j$$

Burada:
- $x_i \in \{0, 1\}$: Gen $i$ seçildi mi? (0=hayır, 1=evet)
- $Q_{ii}$: Gen $i$'nin tek başına katkısı (negatifse 'iyi' gen)
- $Q_{ij}$: Gen $i$ ve $j$'nin birlikte katkısı


In [ ]:
# ============================================================
# ÖRNEK 1: Basit bir QUBO problemi — 3 genle başlayalım
# ============================================================

print("="*60)
print("🧬 ÖRNEK: 3 Gen Arasından En İyi 2'yi Seç")
print("="*60)

# Senaryo:
# Gen A: Hastalıkla zayıf ilişkili (puan = -1)
# Gen B: Hastalıkla güçlü ilişkili (puan = -3)  
# Gen C: Hastalıkla orta ilişkili (puan = -2)
# Gen A+B birlikte seçilirse ceza (puan = +2, çünkü redundant)
# Gen B+C birlikte seçilirse bonus (puan = -1, sinerjik)
# Tam 2 gen seçme kısıtı için ceza ekleyeceğiz

print("\n📊 Gen Bilgileri:")
print("-" * 40)
print(f"  Gen A: Hastalık ilişki skoru = 1 (zayıf)")
print(f"  Gen B: Hastalık ilişki skoru = 3 (güçlü)")
print(f"  Gen C: Hastalık ilişki skoru = 2 (orta)")

# QUBO matrisi oluşturuyoruz
# Sütun/Satır sırası: [A, B, C]

lambda_constraint = 5  # Kısıt ceza katsayısı (yüksek = kısıt daha önemli)
k = 2  # Tam 2 gen seçmek istiyoruz

# Amaç: skoru maksimize et → negatifini minimize et
# Kısıt: (x_A + x_B + x_C - 2)^2 = 0 olmalı
# = x_A^2 + x_B^2 + x_C^2 + 2*x_A*x_B + 2*x_A*x_C + 2*x_B*x_C - 4*x_A - 4*x_B - 4*x_C + 4
# Binary'de x^2 = x olduğundan (0^2=0, 1^2=1):
# = x_A + x_B + x_C + 2*x_A*x_B + 2*x_A*x_C + 2*x_B*x_C - 4*(x_A+x_B+x_C) + 4
# = -3*x_A - 3*x_B - 3*x_C + 2*x_A*x_B + 2*x_A*x_C + 2*x_B*x_C + 4

# QUBO matrisi (Q):
Q = np.zeros((3, 3))

# Diyagonal: amaç + kısıt diyagonal terimleri
scores = np.array([1, 3, 2])  # A, B, C skorları
Q[0, 0] = -scores[0] + lambda_constraint * (1 - 2*k)  # A
Q[1, 1] = -scores[1] + lambda_constraint * (1 - 2*k)  # B  
Q[2, 2] = -scores[2] + lambda_constraint * (1 - 2*k)  # C

# Diyagonal üstü: kısıt çapraz terimleri
Q[0, 1] = lambda_constraint * 2  # A-B
Q[0, 2] = lambda_constraint * 2  # A-C
Q[1, 2] = lambda_constraint * 2  # B-C

print("\n📐 QUBO Matrisi Q:")
print("-" * 40)
gene_names = ['A', 'B', 'C']
df_Q = pd.DataFrame(Q, index=['Gen '+g for g in gene_names], 
                         columns=['Gen '+g for g in gene_names])
print(df_Q.to_string())
print("\n💡 Diyagonal = tek gen puanı, Diyagonal üstü = gen çifti etkileşimi")

In [ ]:
# ============================================================
# QUBO Enerjisini Hesapla: Tüm olası çözümleri dene
# ============================================================

def qubo_energy(x, Q):
    """QUBO enerji fonksiyonu: E = x^T Q x"""
    return float(x @ Q @ x)

print("🔍 Tüm Olası Gen Kombinasyonları ve Enerjileri:")
print("="*60)
print(f"{'Kombinasyon':15} {'x vektörü':15} {'Enerji':10} {'Geçerli mi?':12}")
print("-"*60)

results = []
for bits in range(2**3):  # 3 bit = 8 kombinasyon
    x = np.array([(bits >> i) & 1 for i in range(3)])
    E = qubo_energy(x, Q)
    selected_genes = [gene_names[i] for i in range(3) if x[i] == 1]
    combo_str = '+'.join(selected_genes) if selected_genes else 'Hiçbiri'
    valid = sum(x) == k  # Tam 2 gen seçildi mi?
    results.append((combo_str, x, E, valid))
    
    flag = "✅" if valid else "❌"
    print(f"{combo_str:15} {str(x.tolist()):15} {E:10.2f} {flag}")

# En düşük enerjili geçerli çözümü bul
valid_results = [(c, x, E, v) for c, x, E, v in results if v]
best = min(valid_results, key=lambda r: r[2])

print("\n" + "="*60)
print(f"🏆 En İyi Çözüm: Gen {best[0]}")
print(f"   Enerji: {best[2]:.2f}")
print(f"   Neden? Gen B güçlü (skor=3), Gen C orta (skor=2)")
print("="*60)

In [ ]:
# Enerji manzarasını görselleştir
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

combos = [r[0] for r in results]
energies = [r[2] for r in results]
colors = ['green' if r[3] else 'red' for r in results]

bars = ax1.bar(range(len(combos)), energies, color=colors, edgecolor='black', alpha=0.8)
ax1.set_xticks(range(len(combos)))
ax1.set_xticklabels(combos, rotation=45, ha='right')
ax1.set_xlabel('Gen Kombinasyonu', fontsize=12)
ax1.set_ylabel('QUBO Enerjisi', fontsize=12)
ax1.set_title('QUBO Enerji Manzarası\n(Yeşil = Geçerli çözüm, Kırmızı = Geçersiz)', fontsize=12)
ax1.axhline(y=min(energies), color='gold', linestyle='--', alpha=0.7, label='Min enerji')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# En iyi çözümü işaretle
best_idx = energies.index(min(energies))
ax1.annotate('🏆 En İyi!', xy=(best_idx, energies[best_idx]), 
             xytext=(best_idx+0.5, energies[best_idx]+2),
             arrowprops=dict(arrowstyle='->', color='blue'),
             fontsize=11, color='blue')

# QUBO matrisini görselleştir
im = ax2.imshow(Q, cmap='RdYlGn_r', aspect='auto')
ax2.set_xticks([0,1,2])
ax2.set_yticks([0,1,2])
ax2.set_xticklabels(['Gen A', 'Gen B', 'Gen C'])
ax2.set_yticklabels(['Gen A', 'Gen B', 'Gen C'])
ax2.set_title('QUBO Matrisi (Q)\nKoyu = Yüksek değer', fontsize=12)
plt.colorbar(im, ax=ax2)
for i in range(3):
    for j in range(3):
        ax2.text(j, i, f'{Q[i,j]:.0f}', ha='center', va='center', 
                fontsize=14, fontweight='bold',
                color='white' if abs(Q[i,j]) > 5 else 'black')

plt.tight_layout()
plt.savefig('qubo_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ QUBO görselleştirmesi tamamlandı!")

---
# ⚛️ BÖLÜM 2: Kuantum Bilgisayar Temelleri (Sıfırdan)

## 🎭 Klasik vs Kuantum Bit

| Özellik | Klasik Bit | Kuantum Bit (Qubit) |
|---------|-----------|--------------------|
| Değer | 0 veya 1 | 0 ve 1 **aynı anda** |
| Durum sayısı | 1 | Süperpozisyon |
| Kopyalanabilir mi? | Evet | Hayır (no-cloning) |
| Ölçüm etkisi | Yok | Durumu çöküşe uğratır |

## 🌀 Temel Kavramlar

**Süperpozisyon**: Bir qubit hem |0⟩ hem |1⟩ olabilir:  
$|\psi⟩ = \alpha|0⟩ + \beta|1⟩$, burada $|\alpha|^2 + |\beta|^2 = 1$

**Dolaşıklık (Entanglement)**: İki qubit birbirine bağlı olabilir — birini ölçünce diğeri de belirlenir.

**Pauli Kapıları**: Qubit'leri döndüren temel işlemler

In [ ]:
# ============================================================
# Temel Kuantum Kapıları — Görsel Öğrenme
# ============================================================

print("⚛️  TEMEL KUANTUM KAPILARI")
print("="*50)

# 1. Hadamard (H) Kapısı — Süperpozisyon yaratır
print("\n1️⃣  Hadamard (H) Kapısı — Süperpozisyon yaratır")
print("-"*50)
qc_h = QuantumCircuit(1)
qc_h.h(0)  # Hadamard uygula
print("Devre:")
print(qc_h.draw('text'))

# Durum vektörünü hesapla
sv = Statevector(qc_h)
print(f"\nBaşlangıç: |0⟩ = [1, 0]")
print(f"H sonrası: {sv.data.real.round(3)}")
print(f"Yorum: |+⟩ = (1/√2)|0⟩ + (1/√2)|1⟩")
print(f"0 ölçme olasılığı: {abs(sv.data[0])**2:.2f}")
print(f"1 ölçme olasılığı: {abs(sv.data[1])**2:.2f}")

In [ ]:
# 2. Pauli-X Kapısı — Klasik NOT kapısı gibi
print("2️⃣  Pauli-X Kapısı — Klasik NOT kapısı gibi")
print("-"*50)
qc_x = QuantumCircuit(1)
qc_x.x(0)
print(qc_x.draw('text'))
sv_x = Statevector(qc_x)
print(f"Başlangıç: |0⟩")
print(f"X sonrası: |1⟩ = {sv_x.data.real.round(3)}")

# 3. RZ Kapısı — Açıya göre döndürme
print("\n3️⃣  RZ(θ) Kapısı — Z ekseninde θ kadar döndürme")
print("-"*50)
theta = Parameter('θ')
qc_rz = QuantumCircuit(1)
qc_rz.h(0)  # Önce süperpozisyon
qc_rz.rz(theta, 0)  # Sonra döndür
print(qc_rz.draw('text'))
print("QAOA ve VQC'nin kalbi: parametreli döndürme kapıları!")

# 4. CNOT Kapısı — Dolaşıklık
print("\n4️⃣  CNOT Kapısı — Dolaşıklık (Entanglement)")
print("-"*50)
qc_cnot = QuantumCircuit(2)
qc_cnot.h(0)  # q0'ı süperpozisyona sok
qc_cnot.cx(0, 1)  # q0 kontrol, q1 hedef
print(qc_cnot.draw('text'))
sv_bell = Statevector(qc_cnot)
print(f"\nBell durumu (dolaşık): {sv_bell.data.round(3)}")
print("Yorum: (1/√2)|00⟩ + (1/√2)|11⟩")
print("q0 ölçülünce q1 de belirlenir — bu dolaşıklık!")

In [ ]:
# Kapıların Bloch Küre üzerinde görselleştirilmesi
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

kapılar = [
    ('|0⟩ (Başlangıç)', [0, 0, 1], 'blue'),
    ('H|0⟩ = |+⟩', [1, 0, 0], 'green'),
    ('X|0⟩ = |1⟩', [0, 0, -1], 'red'),
]

for ax, (isim, bloch_vec, renk) in zip(axes, kapılar):
    # Basit Bloch küre çizimi
    theta_plot = np.linspace(0, np.pi, 50)
    phi_plot = np.linspace(0, 2*np.pi, 50)
    T, P = np.meshgrid(theta_plot, phi_plot)
    x_s = np.sin(T) * np.cos(P)
    y_s = np.sin(T) * np.sin(P)
    z_s = np.cos(T)
    
    ax.set_aspect('equal')
    circle_xy = plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--', alpha=0.5)
    ax.add_patch(circle_xy)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    
    # Eksenler
    ax.axhline(y=0, color='gray', alpha=0.3)
    ax.axvline(x=0, color='gray', alpha=0.3)
    
    # Kutuplar
    ax.text(0, 1.2, '|0⟩', ha='center', fontsize=12, fontweight='bold')
    ax.text(0, -1.2, '|1⟩', ha='center', fontsize=12, fontweight='bold')
    ax.text(1.2, 0, '|+⟩', ha='center', fontsize=10, color='green')
    ax.text(-1.2, 0, '|-⟩', ha='center', fontsize=10, color='green')
    
    # Kuantum durumu vektörü
    ax.annotate('', xy=(bloch_vec[0], bloch_vec[2]), xytext=(0, 0),
               arrowprops=dict(arrowstyle='->', color=renk, lw=3))
    ax.scatter([bloch_vec[0]], [bloch_vec[2]], color=renk, s=100, zorder=5)
    
    ax.set_title(isim, fontsize=12, fontweight='bold')
    ax.set_xlabel('X (|+⟩ yönü)', fontsize=9)
    ax.set_ylabel('Z (|0⟩ yönü)', fontsize=9)

plt.suptitle('Bloch Küre: Qubit Durumlarının Görselleştirilmesi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('bloch_sphere.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Bloch küre görselleştirmesi tamamlandı!")

---
# 🌊 BÖLÜM 3: QAOA — Kuantum Yaklaşık Optimizasyon Algoritması

## 🧠 Sezgi: QAOA Nasıl Çalışır?

QAOA'yı şöyle düşünün: bir dağlık arazide en derin vadiyi (minimum enerji) arıyorsunuz.

**Klasik yöntem**: Her noktaya tek tek bakın → yavaş  
**QAOA**: Tüm noktaları **aynı anda** (süperpozisyon ile) kontrol edin → hızlı

## 📐 QAOA'nın Yapısı

QAOA iki tip kapıyı dönüşümlü uygular:

1. **Maliyet katmanı** (Problem Hamiltonian $U_C$): QUBO'nun kuantum versiyonu  
   $U_C(\gamma) = e^{-i\gamma H_C}$ — problemi kodlar

2. **Karıştırıcı katman** (Mixer Hamiltonian $U_B$): Çözümleri karıştırır  
   $U_B(\beta) = e^{-i\beta H_B}$ — uzay keşfeder

**p katman** sonrası: $|\psi(\gamma, \beta)⟩ = U_B(\beta_p) U_C(\gamma_p) \cdots U_B(\beta_1) U_C(\gamma_1) |+⟩^{\otimes n}$

**Klasik optimizer**: $\gamma$ ve $\beta$ değerlerini ayarlar → beklenti değerini minimize eder

In [ ]:
# ============================================================
# QAOA'yı SIFIRDAN İNŞA ET — En Basit Örnekle
# Problem: 2 gen arasından birini seç (Max-Cut benzeri)
# ============================================================

print("🌊 QAOA SIFIRDAN: 2 Gen Seçim Problemi")
print("="*60)
print("Problem: 2 gen var, hangisi daha önemli?")
print("Gen 0 (BRCA1): puan = -2 (çok önemli)")
print("Gen 1 (TP53): puan = -1 (önemli)")
print("Hedef: Sadece 1 gen seç, en düşük enerjiliyi bul")
print("-"*60)

# Adım 1: QUBO'yu Ising modeline dönüştür
# QUBO: x ∈ {0,1} — Ising: z ∈ {-1,+1}
# Dönüşüm: x_i = (1 - z_i) / 2

# Basit problem: minimize -2*x0 - 1*x1 + 3*(x0+x1-1)^2
# Constraint: tam 1 gen seç
# lambda_c * (x0 + x1 - 1)^2 = lambda_c*(x0^2 + x1^2 - 2x0 - 2x1 + 2x0x1 + 1)
# = lambda_c*(x0 + x1 - 2x0 - 2x1 + 2x0x1 + 1)  [x^2=x için binary'de]
# = lambda_c*(-x0 - x1 + 2x0x1 + 1)

# Toplam: (-2 - lambda_c)*x0 + (-1 - lambda_c)*x1 + 2*lambda_c*x0*x1 + const

lambda_c = 3
Q_2gen = np.array([
    [-2 - lambda_c,  lambda_c],   # [Q00, Q01]
    [0,             -1 - lambda_c] # [Q10, Q11]
])

print("\n📐 2-Gen QUBO Matrisi:")
print(Q_2gen)

# Brute-force kontrol
print("\n🔍 Brute-Force Doğrulama:")
for x0 in [0, 1]:
    for x1 in [0, 1]:
        x = np.array([x0, x1])
        E = float(x @ Q_2gen @ x)
        valid = (x0 + x1) == 1
        flag = "✅" if valid else "❌"
        print(f"  x=[{x0},{x1}]: Enerji={E:.1f} {flag}")

In [ ]:
# ============================================================
# QAOA Devresini İnşa Et — Adım Adım
# ============================================================

def build_qaoa_circuit(gamma, beta, p=1):
    """
    2-qubit QAOA devresi oluşturur.
    
    Parametreler:
    - gamma: Maliyet katmanı açısı (liste, p tane)
    - beta: Karıştırıcı katman açısı (liste, p tane)
    - p: QAOA katman sayısı
    """
    n_qubits = 2
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # ADIM 1: Başlangıç durumu — eşit süperpozisyon
    # |+⟩^⊗n = H|0⟩^⊗n
    qc.h(range(n_qubits))
    qc.barrier(label='Başlangıç: |+⟩⊗|+⟩')
    
    for layer in range(p):
        # ADIM 2: Maliyet katmanı U_C(gamma)
        # Problem Hamiltonian'ını uygula
        # H_C = Q_00*Z_0 + Q_11*Z_1 + Q_01*Z_0*Z_1 + const
        # (Ising dönüşümü ile)
        
        # Tek qubit terimleri: RZ(2*gamma*h_i)
        # Çift qubit terimleri: CNOT + RZ + CNOT
        
        # Önce Ising katsayılarını hesapla
        # x_i = (1 - z_i)/2 dönüşümü ile:
        # Q00*x0 = Q00*(1-z0)/2 → z0'a katkı: -Q00/2
        # Q01*x0*x1 = Q01*(1-z0)(1-z1)/4 → z0z1'e katkı: Q01/4
        
        h0 = -Q_2gen[0,0] / 2  # z0 katsayısı
        h1 = -Q_2gen[1,1] / 2  # z1 katsayısı
        J01 = Q_2gen[0,1] / 4  # z0*z1 katsayısı
        
        # RZ kapıları (tek qubit)
        qc.rz(2 * gamma[layer] * h0, 0)
        qc.rz(2 * gamma[layer] * h1, 1)
        
        # ZZ etkileşimi (çift qubit)
        qc.cx(0, 1)
        qc.rz(2 * gamma[layer] * J01, 1)
        qc.cx(0, 1)
        
        qc.barrier(label=f'Maliyet γ{layer+1}')
        
        # ADIM 3: Karıştırıcı katman U_B(beta)
        # H_B = Σ_i X_i → RX(2*beta) her qubit'e
        for qubit in range(n_qubits):
            qc.rx(2 * beta[layer], qubit)
        
        qc.barrier(label=f'Karıştırıcı β{layer+1}')
    
    # ADIM 4: Ölçüm
    qc.measure(range(n_qubits), range(n_qubits))
    
    return qc

# Test devresini çiz
test_circuit = build_qaoa_circuit([0.5], [0.3], p=1)
print("🔌 QAOA Devresi (p=1):")
print(test_circuit.draw('text'))
print("\n📊 Devre İstatistikleri:")
print(f"  Qubit sayısı: {test_circuit.num_qubits}")
print(f"  Kapı sayısı: {test_circuit.size()}")
print(f"  Derinlik: {test_circuit.depth()}")

In [ ]:
# ============================================================
# QAOA Optimizasyonu — Klasik Optimizer ile
# ============================================================

from scipy.optimize import minimize

def qaoa_expectation(params):
    """
    Verilen parametreler için QAOA beklenti değerini hesapla.
    Bu fonksiyon klasik optimizer tarafından minimize edilir.
    """
    p = 1
    gamma = [params[0]]
    beta = [params[1]]
    
    # Devreyi oluştur ve simüle et
    qc = build_qaoa_circuit(gamma, beta, p)
    
    # Simülatörü çalıştır
    simulator = AerSimulator()
    from qiskit import transpile
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled, shots=1024)
    result = job.result()
    counts = result.get_counts()
    
    # Beklenti değeri: Σ P(x) * E(x)
    total_shots = sum(counts.values())
    expectation = 0
    
    for bitstring, count in counts.items():
        # Qiskit'te bitstring ters sıralı: '10' → x0=0, x1=1
        x = np.array([int(b) for b in reversed(bitstring)])
        E = float(x @ Q_2gen @ x)
        prob = count / total_shots
        expectation += prob * E
    
    return expectation

# Optimizasyonu çalıştır
print("🔧 QAOA Optimizasyonu Başlıyor...")
print("-"*50)

optimization_history = []

def callback(params):
    val = qaoa_expectation(params)
    optimization_history.append(val)

# Başlangıç parametreleri
x0 = np.array([0.1, 0.1])

result_opt = minimize(
    qaoa_expectation, 
    x0, 
    method='COBYLA',
    options={'maxiter': 100, 'rhobeg': 0.5},
    callback=callback
)

print(f"✅ Optimizasyon tamamlandı!")
print(f"   En iyi γ = {result_opt.x[0]:.4f}")
print(f"   En iyi β = {result_opt.x[1]:.4f}")
print(f"   Minimum beklenti değeri = {result_opt.fun:.4f}")

In [ ]:
# Optimal parametrelerle final ölçüm
optimal_gamma = [result_opt.x[0]]
optimal_beta = [result_opt.x[1]]

qc_optimal = build_qaoa_circuit(optimal_gamma, optimal_beta, p=1)

simulator = AerSimulator()
from qiskit import transpile
compiled = transpile(qc_optimal, simulator)
job = simulator.run(compiled, shots=4096)
counts_optimal = job.result().get_counts()

print("📊 Optimal QAOA Ölçüm Sonuçları:")
print("-"*50)
total = sum(counts_optimal.values())
for bitstring, count in sorted(counts_optimal.items(), key=lambda x: -x[1]):
    x = np.array([int(b) for b in reversed(bitstring)])
    E = float(x @ Q_2gen @ x)
    prob = count / total
    genes = [f'Gen{i}' for i in range(2) if x[i] == 1]
    gene_str = '+'.join(genes) if genes else 'Hiçbiri'
    bar = '█' * int(prob * 40)
    print(f"  |{bitstring}⟩ ({gene_str:12}): {bar:40} {prob:.1%} (E={E:.1f})")

# Görselleştir
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Optimizasyon geçmişi
if optimization_history:
    ax1.plot(optimization_history, 'b-o', markersize=4, alpha=0.7)
    ax1.set_xlabel('İterasyon', fontsize=12)
    ax1.set_ylabel('Beklenti Değeri ⟨E⟩', fontsize=12)
    ax1.set_title('QAOA Optimizasyon Sürecı', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=result_opt.fun, color='red', linestyle='--', label=f'Min = {result_opt.fun:.2f}')
    ax1.legend()

# Final olasılıklar
keys = list(counts_optimal.keys())
probs = [counts_optimal[k]/total for k in keys]
gene_labels = []
colors_bar = []
for k in keys:
    x = np.array([int(b) for b in reversed(k)])
    genes = [f'Gen{i}' for i in range(2) if x[i] == 1]
    gene_labels.append('+'.join(genes) if genes else 'Yok')
    E = float(x @ Q_2gen @ x)
    colors_bar.append('green' if sum(x) == 1 else 'red')

bars = ax2.bar(range(len(keys)), probs, color=colors_bar, alpha=0.8, edgecolor='black')
ax2.set_xticks(range(len(keys)))
ax2.set_xticklabels([f'|{k}⟩\n{gene_labels[i]}' for i, k in enumerate(keys)])
ax2.set_ylabel('Olasılık', fontsize=12)
ax2.set_title('QAOA Final Ölçüm Dağılımı\n(Yeşil = Geçerli çözüm)', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('qaoa_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n🎯 En yüksek olasılıklı geçerli çözüm = QAOA'nın önerisi!")

---
# 🔬 BÖLÜM 4: VQC — Değişken Kuantum Devresi

## VQC vs QAOA: Fark Ne?

| Özellik | QAOA | VQC |
|---------|------|-----|
| Yapı | Problem'e göre tasarlanmış | Genel amaçlı |
| Katmanlar | Maliyet + Karıştırıcı | Herhangi devre |
| Optimize edilen | γ ve β açıları | Tüm kapı açıları |
| Kullanım | Kombinatoryal opt. | Makine öğrenmesi, sınıflandırma |
| Q-OmicSelect'te | QUBO çözümü | Gen özellik çıkarımı |

## VQC Yapısı

1. **Veri kodlama katmanı**: Klasik verileri qubitlere kodla
2. **Parametreli katmanlar**: Öğrenilebilir açılar ($\theta$)
3. **Ölçüm**: Çıktıyı sınıflandır

In [ ]:
# ============================================================
# VQC — Gen Ekspresyonu Sınıflandırması
# Problem: 2 gen ekspresyon değeri → Hasta mı/Sağlıklı mı?
# ============================================================

print("🔬 VQC: Gen Ekspresyonu Sınıflandırması")
print("="*60)
print("Giriş: 2 genin normalized ekspresyon değerleri")
print("Çıkış: Hasta (1) veya Sağlıklı (0)")
print("-"*60)

# Sentetik veri oluştur
np.random.seed(42)
n_samples = 40

# Sağlıklı: düşük ekspresyon
X_healthy = np.random.normal([0.2, 0.3], 0.1, (n_samples//2, 2))
y_healthy = np.zeros(n_samples//2)

# Hasta: yüksek ekspresyon  
X_sick = np.random.normal([0.7, 0.8], 0.1, (n_samples//2, 2))
y_sick = np.ones(n_samples//2)

X = np.clip(np.vstack([X_healthy, X_sick]), 0, 1)
y = np.hstack([y_healthy, y_sick])

print(f"\n📊 Veri seti:")
print(f"  Toplam örnek: {len(y)}")
print(f"  Sağlıklı: {sum(y==0)} örnek")
print(f"  Hasta: {sum(y==1)} örnek")
print(f"  Özellik (gen) sayısı: 2")

# VQC Devresi
def build_vqc(x_data, theta):
    """
    Basit VQC devresi.
    
    x_data: [gene1_expr, gene2_expr] — veri kodlama
    theta: [θ1, θ2, θ3, θ4] — öğrenilebilir parametreler
    """
    qc = QuantumCircuit(2, 1)
    
    # --- KATMAN 1: Veri Kodlama ---
    # Angle encoding: ekspresyon değeri → açı
    qc.ry(np.pi * x_data[0], 0)  # Gen1 → qubit 0
    qc.ry(np.pi * x_data[1], 1)  # Gen2 → qubit 1
    qc.barrier(label='Veri')
    
    # --- KATMAN 2: Parametreli Katman 1 ---
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)  # Dolaşıklık
    qc.barrier(label='L1')
    
    # --- KATMAN 3: Parametreli Katman 2 ---
    qc.ry(theta[2], 0)
    qc.ry(theta[3], 1)
    qc.barrier(label='L2')
    
    # --- ÖLÇÜM ---
    qc.measure(0, 0)  # Sadece qubit 0'ı ölç
    
    return qc

# Örnek VQC devresini göster
sample_theta = np.random.uniform(0, 2*np.pi, 4)
sample_x = [0.3, 0.7]
vqc_demo = build_vqc(sample_x, sample_theta)
print("\n🔌 VQC Devresi Yapısı:")
print(vqc_demo.draw('text'))

In [ ]:
# VQC Eğitimi
simulator = AerSimulator()

def vqc_predict(x_data, theta, shots=512):
    """VQC ile tahmin yap: 0 veya 1 döndür"""
    qc = build_vqc(x_data, theta)
    from qiskit import transpile
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled, shots=shots)
    counts = job.result().get_counts()
    prob_1 = counts.get('1', 0) / shots
    return prob_1

def vqc_loss(theta, X_train, y_train, shots=256):
    """Binary cross-entropy kaybı"""
    total_loss = 0
    for x, y_true in zip(X_train, y_train):
        p = vqc_predict(x, theta, shots)
        p = np.clip(p, 1e-7, 1-1e-7)
        loss = -y_true * np.log(p) - (1-y_true) * np.log(1-p)
        total_loss += loss
    return total_loss / len(y_train)

# Küçük subset ile eğit (Colab için hızlı)
print("🏋️ VQC Eğitimi Başlıyor (bu ~1-2 dakika sürer)...")
print("-"*50)

# Küçük eğitim seti kullan
train_size = 16
idx = np.random.choice(len(X), train_size, replace=False)
X_train = X[idx]
y_train = y[idx]

# Başlangıç parametreleri
theta_init = np.random.uniform(0, 2*np.pi, 4)

vqc_history = []
def vqc_callback(params):
    loss = vqc_loss(params, X_train, y_train, shots=128)
    vqc_history.append(loss)
    if len(vqc_history) % 10 == 0:
        print(f"  İterasyon {len(vqc_history):3d}: Kayıp = {loss:.4f}")

result_vqc = minimize(
    vqc_loss,
    theta_init,
    args=(X_train, y_train, 128),
    method='COBYLA',
    options={'maxiter': 50, 'rhobeg': 0.3},
    callback=vqc_callback
)

print(f"\n✅ Eğitim tamamlandı!")
print(f"   Final kayıp: {result_vqc.fun:.4f}")

In [ ]:
# VQC Test ve Görselleştirme
print("🧪 VQC Test Sonuçları:")
print("-"*50)

optimal_theta = result_vqc.x
predictions = []
test_indices = np.random.choice(len(X), 20, replace=False)

correct = 0
for i in test_indices:
    prob = vqc_predict(X[i], optimal_theta, shots=512)
    pred = 1 if prob > 0.5 else 0
    predictions.append(pred)
    if pred == y[i]:
        correct += 1

accuracy = correct / len(test_indices)
print(f"  Test doğruluğu: {accuracy:.1%} ({correct}/{len(test_indices)})")

# Görselleştir
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Eğitim geçmişi
if vqc_history:
    ax1.plot(vqc_history, 'g-o', markersize=4)
    ax1.set_xlabel('İterasyon', fontsize=12)
    ax1.set_ylabel('Kayıp (Cross-Entropy)', fontsize=12)
    ax1.set_title('VQC Eğitim Geçmişi', fontsize=12)
    ax1.grid(True, alpha=0.3)

# Veri ve karar sınırı
ax2.scatter(X_healthy[:, 0], X_healthy[:, 1], c='blue', alpha=0.7, label='Sağlıklı', s=50)
ax2.scatter(X_sick[:, 0], X_sick[:, 1], c='red', alpha=0.7, label='Hasta', s=50)

# Karar sınırı
xx = np.linspace(0, 1, 15)
yy = np.linspace(0, 1, 15)
Z = np.zeros((15, 15))
for i, xi in enumerate(xx):
    for j, yj in enumerate(yy):
        Z[j, i] = vqc_predict([xi, yj], optimal_theta, shots=128)

ax2.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu_r', levels=10)
ax2.set_xlabel('Gen 1 Ekspresyonu', fontsize=12)
ax2.set_ylabel('Gen 2 Ekspresyonu', fontsize=12)
ax2.set_title(f'VQC Karar Sınırı\nDoğruluk: {accuracy:.1%}', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vqc_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ VQC görselleştirmesi tamamlandı!")

---
# 🧬 BÖLÜM 5: Omik Veri ile Büyük Resim — Q-OmicSelect Prototipi

## Şimdiye kadar öğrendiklerimizi birleştirelim:

```
GEN EKSPRESYON VERİSİ
       ↓
CPO/GWO (Wrapper Feature Selection)
       ↓ Aday gen subsetleri
QUBO Formülasyonu
       ↓ Q matrisi
QAOA/VQC (Kuantum Alt-Rutin)
       ↓ En iyi subset
LLM Ajan (Biyolojik yorum)
       ↓
SONUÇ: Optimize gen seti
```

In [ ]:
# ============================================================
# MİNİ Q-OmicSelect Prototipi
# Sentetik scRNA-seq benzeri veri ile
# ============================================================

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif

print("🧬 MİNİ Q-OmicSelect Prototipi")
print("="*60)

# Adım 1: Sentetik gen ekspresyon verisi oluştur
print("\n📊 Adım 1: Sentetik Gen Ekspresyonu Verisi")
np.random.seed(42)

n_genes = 10  # Toplam gen sayısı
n_informative = 4  # Gerçekten önemli gen sayısı
n_samples_omics = 100

X_genes, y_disease = make_classification(
    n_samples=n_samples_omics,
    n_features=n_genes,
    n_informative=n_informative,
    n_redundant=2,
    random_state=42
)

gene_names_omics = [f'GENE_{i+1:02d}' for i in range(n_genes)]
df_expr = pd.DataFrame(X_genes, columns=gene_names_omics)

print(f"  Veri boyutu: {X_genes.shape}")
print(f"  Gen sayısı: {n_genes}")
print(f"  Örnek sayısı: {n_samples_omics}")
print(f"  Bilgilendirici gen sayısı: {n_informative} (gizli)")
print("\nİlk 5 satır:")
print(df_expr.head().round(3).to_string())

In [ ]:
# Adım 2: Mutual Information ile gen ön-sıralaması (CPO proxy)
print("\n🔍 Adım 2: Mutual Information Tabanlı Ön-Filtreleme")
print("-"*50)
print("(Gerçek Q-OmicSelect'te bu CPO/GWO ile yapılır)")

mi_scores = mutual_info_classif(X_genes, y_disease, random_state=42)
gene_scores = pd.Series(mi_scores, index=gene_names_omics).sort_values(ascending=False)

print("\nGen Önem Skorları (Mutual Information):")
for gene, score in gene_scores.items():
    bar = '█' * int(score * 50)
    print(f"  {gene}: {bar:30} {score:.4f}")

# Top 4 geni seç (QUBO için)
top_genes = gene_scores.index[:4].tolist()
print(f"\n🎯 QUBO için seçilen top {len(top_genes)} gen: {top_genes}")

In [ ]:
# Adım 3: QUBO Formülasyonu
print("\n📐 Adım 3: QUBO Formülasyonu")
print("-"*50)

def build_gene_qubo(gene_list, mi_scores_dict, X_data, k_select=2, lambda_penalty=5):
    """
    Gen seçimi için QUBO matrisi oluşturur.
    
    Amaç fonksiyonu:
    - MI skoru yüksek genleri seç (negatif diyagonal)
    - Korelasyonlu gen çiftlerini penalize et (pozitif çift terim)
    - Tam k gen seç (kısıt)
    """
    n = len(gene_list)
    Q = np.zeros((n, n))
    
    # Korelasyon matrisi
    X_subset = X_data[:, [gene_names_omics.index(g) for g in gene_list]]
    corr_matrix = np.corrcoef(X_subset.T)
    
    # Diyagonal: -MI skoru + kısıt diyagonal terimi
    for i, gene in enumerate(gene_list):
        mi = mi_scores_dict[gene]
        Q[i, i] = -mi + lambda_penalty * (1 - 2*k_select)
    
    # Üst diyagonal: korelasyon cezası + kısıt çift terimi
    for i in range(n):
        for j in range(i+1, n):
            corr_penalty = abs(corr_matrix[i, j])  # Korelasyon → ceza
            Q[i, j] = corr_penalty + lambda_penalty * 2
    
    return Q, corr_matrix

mi_dict = gene_scores.to_dict()
Q_omics, corr_mat = build_gene_qubo(top_genes, mi_dict, X_genes, k_select=2)

print("QUBO Matrisi (4x4):")
df_qubo = pd.DataFrame(Q_omics, index=top_genes, columns=top_genes)
print(df_qubo.round(3).to_string())

print("\nKorelasyon Matrisi:")
df_corr = pd.DataFrame(corr_mat, index=top_genes, columns=top_genes)
print(df_corr.round(3).to_string())

In [ ]:
# Adım 4: Klasik çözüm (doğrulama için)
print("\n🔎 Adım 4: Brute-Force Doğrulama")
print("-"*50)

best_combo = None
best_energy = float('inf')

print(f"{'Kombinasyon':20} {'Enerji':10} {'Geçerli?':10}")
print("-"*45)

for bits in range(2**4):
    x = np.array([(bits >> i) & 1 for i in range(4)])
    E = float(x @ Q_omics @ x)
    selected = [top_genes[i] for i in range(4) if x[i] == 1]
    valid = sum(x) == 2
    combo_str = '+'.join([g.replace('GENE_', 'G') for g in selected]) if selected else 'Yok'
    flag = "✅" if valid else "  "
    if valid and E < best_energy:
        best_energy = E
        best_combo = selected
    if valid:
        print(f"{combo_str:20} {E:10.3f} {flag}")

print(f"\n🏆 En İyi Kombinasyon: {best_combo}")
print(f"   Enerji: {best_energy:.3f}")

In [ ]:
# Final görselleştirme — Tüm pipeline
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Gen ekspresyon heatmap
ax = axes[0, 0]
sample_expr = df_expr.head(20)
im = ax.imshow(sample_expr.T.values, aspect='auto', cmap='RdYlGn')
ax.set_yticks(range(n_genes))
ax.set_yticklabels(gene_names_omics, fontsize=8)
ax.set_xlabel('Örnek (Hücre)', fontsize=10)
ax.set_title('Gen Ekspresyon Isı Haritası\n(İlk 20 örnek)', fontsize=10)
plt.colorbar(im, ax=ax, label='Ekspresyon')

# 2. MI skorları
ax = axes[0, 1]
colors_mi = ['darkgreen' if g in top_genes else 'gray' for g in gene_scores.index]
ax.barh(range(n_genes), gene_scores.values, color=colors_mi, alpha=0.8)
ax.set_yticks(range(n_genes))
ax.set_yticklabels(gene_scores.index, fontsize=9)
ax.set_xlabel('Mutual Information Skoru', fontsize=10)
ax.set_title('Gen Önem Skorları\n(Koyu yeşil = QUBO\'ya dahil)', fontsize=10)
ax.axvline(x=gene_scores.values[3], color='red', linestyle='--', alpha=0.7, label='Eşik')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# 3. QUBO matrisi
ax = axes[1, 0]
im2 = ax.imshow(Q_omics, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels([g.replace('GENE_','G') for g in top_genes])
ax.set_yticklabels([g.replace('GENE_','G') for g in top_genes])
ax.set_title('QUBO Matrisi\n(Omik veriden türetildi)', fontsize=10)
plt.colorbar(im2, ax=ax)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{Q_omics[i,j]:.1f}', ha='center', va='center',
               fontsize=9, color='white' if abs(Q_omics[i,j]) > 6 else 'black')

# 4. Pipeline özeti
ax = axes[1, 1]
ax.axis('off')
pipeline_text = [
    ('📊 VERİ', f'{n_genes} gen × {n_samples_omics} hücre', 'royalblue'),
    ('🔍 FİLTRE', f'MI → Top {len(top_genes)} gen', 'forestgreen'),
    ('📐 QUBO', f'4×4 Q matrisi (λ=5)', 'darkorange'),
    ('⚛️ QAOA', f'p=1, 2 qubit optimizasyonu', 'purple'),
    ('🏆 SONUÇ', f'{" + ".join([g.replace("GENE_","G") for g in best_combo])}', 'crimson'),
]

for idx, (title, desc, color) in enumerate(pipeline_text):
    y_pos = 0.85 - idx * 0.18
    ax.text(0.05, y_pos, title, fontsize=13, fontweight='bold', 
           color=color, transform=ax.transAxes)
    ax.text(0.35, y_pos, desc, fontsize=11, color='black', transform=ax.transAxes)
    if idx < len(pipeline_text) - 1:
        ax.annotate('', xy=(0.15, y_pos - 0.12), xytext=(0.15, y_pos - 0.04),
                   xycoords='axes fraction', textcoords='axes fraction',
                   arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('Q-OmicSelect Pipeline', fontsize=12, fontweight='bold')

plt.suptitle('Mini Q-OmicSelect: Tam Pipeline Özeti', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('qomicselect_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Q-OmicSelect prototipi tamamlandı!")

---
# 🎓 BÖLÜM 6: Öğrenme Özeti ve Sonraki Adımlar

## ✅ Bu Notebook'ta Öğrendikleriniz

| Konu | Öğrendiğiniz |
|------|-------------|
| **QUBO** | Binary optimizasyon problemlerini matris formunda ifade etme |
| **Kuantum kapıları** | H, X, RZ, RY, CNOT kapılarının işlevi |
| **QAOA** | Problem Hamiltonian + Mixer = Optimizasyon |
| **VQC** | Parametreli devreler ile makine öğrenmesi |
| **Omik bağlantısı** | Gen ekspresyonunu QUBO'ya dönüştürme |
| **Pipeline** | Tüm bileşenleri bir araya getirme |

## 🚀 Sonraki Adımlar

1. **QAOA'yı daha derin yapın**: p=2, p=3 katmanlar deneyin
2. **Daha büyük problem**: 6-8 gen ile deneyin
3. **Gerçek veri**: GEO'dan scRNA-seq verisi indirin
4. **CPO/GWO entegrasyonu**: Wrapper seçim algoritmaları ekleyin
5. **LLM ajan**: OpenAI/Anthropic API ile biyolojik yorum ekleyin

In [ ]:
# ============================================================
# BONUS: QAOA'yı Derinleştir — p=2 Katman
# ============================================================

print("🚀 BONUS: QAOA p=2 Katman")
print("="*50)
print("Daha fazla katman → daha iyi optimizasyon")

def qaoa_expectation_p2(params):
    """p=2 QAOA için beklenti değeri"""
    gamma = [params[0], params[2]]
    beta = [params[1], params[3]]
    
    qc = build_qaoa_circuit(gamma, beta, p=2)
    simulator = AerSimulator()
    from qiskit import transpile
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled, shots=512)
    counts = job.result().get_counts()
    
    total = sum(counts.values())
    expectation = 0
    for bitstring, count in counts.items():
        x = np.array([int(b) for b in reversed(bitstring)])
        E = float(x @ Q_2gen @ x)
        expectation += (count / total) * E
    return expectation

# p=1 vs p=2 karşılaştırması
print("\n📊 p=1 vs p=2 Karşılaştırması:")

# p=2 optimizasyonu
x0_p2 = np.random.uniform(0, np.pi, 4)
result_p2 = minimize(
    qaoa_expectation_p2, x0_p2,
    method='COBYLA',
    options={'maxiter': 80, 'rhobeg': 0.5}
)

print(f"  p=1 minimum beklenti: {result_opt.fun:.4f}")
print(f"  p=2 minimum beklenti: {result_p2.fun:.4f}")
print(f"  İyileşme: {result_opt.fun - result_p2.fun:.4f}")
print("\n💡 Daha fazla katman → Gerçek minimum'a daha yakın!")
print("   Ama daha fazla hesaplama süresi gerektirir.")

print("\n" + "="*60)
print("🎉 TEBRIKLER! Notebook tamamlandı.")
print("Q-OmicSelect yolculuğunuza iyi başarılar! 🧬⚛️")
print("="*60)